In [1]:
%pip install -q datasets huggingface_hub pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import re
import hashlib
import unicodedata

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import HfApi

DATASET_ID = "tmquan/cbba-toaan-gov-vn"

TARGET = 5000         # Tổng số bản ứng viên muốn lưu
MAX_SCAN = 1_542_247     # Số bản ghi tối đa quét trong mỗi lượt

OUT = Path("dataset_dan_su_dat_dai")
OUT.mkdir(exist_ok=True)

RAW_FILE = OUT / "candidates_raw.jsonl"
CONFIG_FILE = OUT / "collection_config.json"

# Giữ cố định phiên bản dataset để lần chạy sau không đổi nguồn.
if CONFIG_FILE.exists():
    config = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
    REVISION = config["revision"]
else:
    REVISION = HfApi().dataset_info(DATASET_ID).sha

    config = {
        "dataset_id": DATASET_ID,
        "revision": REVISION,
        "purpose": "Thu ứng viên dân sự liên quan đất đai; chưa gán nhãn kết quả"
    }
    CONFIG_FILE.write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

print("Thư mục lưu:", OUT.resolve())
print("Phiên bản nguồn:", REVISION)

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thư mục lưu: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\notebooks\dataset_dan_su_dat_dai
Phiên bản nguồn: 8ace3afc9c974afd51e12dccbefd4b3ba54ce1ec


In [3]:
def normalize(text):
    """Chuyển về chữ thường, bỏ dấu để tìm từ khóa ổn định hơn."""
    text = unicodedata.normalize("NFD", str(text or "").lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = text.replace("đ", "d")
    return re.sub(r"\s+", " ", text).strip()


KEYWORDS = {
    "loi_di_bds_lien_ke": [
        "lối đi chung", "lối đi qua", "tranh chấp lối đi",
        "mở lối đi", "ngõ đi chung", "bất động sản liền kề"
    ],
    "chuyen_nhuong_qsdd": [
        "hợp đồng chuyển nhượng quyền sử dụng đất",
        "hợp đồng chuyển nhượng đất",
        "chuyển nhượng quyền sử dụng đất"
    ],
    "doi_nha_dat_o_nho": [
        "đất cho ở nhờ", "nhà cho ở nhờ",
        "đòi lại nhà đất", "đòi nhà cho ở nhờ",
        "đòi lại đất cho ở nhờ"
    ],
    "ranh_gioi_lan_chiem": [
        "tranh chấp ranh giới", "ranh giới quyền sử dụng đất",
        "đất lấn chiếm", "lấn chiếm đất"
    ],
    "dat_dai_khac": [
        "tranh chấp quyền sử dụng đất",
        "tranh chấp đất đai",
        "đòi lại quyền sử dụng đất",
        "hợp đồng đặt cọc chuyển nhượng",
        "thừa kế quyền sử dụng đất"
    ]
}

NORMALIZED_KEYWORDS = {
    group: [normalize(term) for term in terms]
    for group, terms in KEYWORDS.items()
}


def screen_record(row):
    text = str(row.get("markdown") or "")
    category = normalize(row.get("category"))
    code = normalize(row.get("code")).replace(" ", "")

    # Giữ Civil; nếu thiếu category thì dùng mã DS làm dấu hiệu dự phòng.
    is_civil = category in {"civil", "dan su"}

    if not category:
        is_civil = "ds" in code and "hngd" not in code

    if not is_civil or len(text.strip()) < 1500:
        return None

    # Tìm ở phần đầu bản án để giảm trúng từ khóa chỉ có trong viện dẫn luật.
    head = normalize(text[:12000])

    groups = []
    matched_terms = []

    for group, terms in NORMALIZED_KEYWORDS.items():
        hits = [term for term in terms if term in head]
        if hits:
            groups.append(group)
            matched_terms.extend(hits)

    if not groups:
        return None

    # Mỗi bản chỉ lưu một lần, nhưng có thể thuộc nhiều nhóm.
    return {
        "candidate_groups": groups,
        "matched_terms": sorted(set(matched_terms)),
        "review_status": "pending_manual_review"
    }


print("Đã chuẩn bị bộ lọc.")

Đã chuẩn bị bộ lọc.


In [7]:
from huggingface_hub import HfApi, hf_hub_url


def text_hash(text):
    return hashlib.sha256(
        normalize(text).encode("utf-8")
    ).hexdigest()


# Đọc các bản đã lưu.
seen_ids = set()
seen_hashes = set()
saved = 0

if RAW_FILE.exists():
    with RAW_FILE.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"File lưu có dòng JSON lỗi ở dòng {line_number}. "
                    "Dừng để tránh ghi thêm vào file lỗi."
                ) from error

            source = record["source_record"]
            doc_id = str(source.get("doc_name") or "")

            if doc_id:
                seen_ids.add(doc_id)

            seen_hashes.add(record["text_sha256"])
            saved += 1

print("Đã lưu từ trước:", saved)

if saved >= TARGET:
    print("Đã đạt TARGET, không cần tải thêm.")

else:
    inspected = 0
    added = 0

    try:
        # Chỉ lấy các file nội dung bản án.
        repo_files = HfApi().list_repo_files(
            DATASET_ID,
            repo_type="dataset",
            revision=REVISION
        )

        parquet_files = sorted(
            name for name in repo_files
            if name.startswith("documents-")
            and name.endswith(".parquet")
        )

        if not parquet_files:
            raise RuntimeError(
                "Không tìm thấy file documents-*.parquet."
            )

        urls = [
            hf_hub_url(
                repo_id=DATASET_ID,
                filename=name,
                repo_type="dataset",
                revision=REVISION
            )
            for name in parquet_files
        ]

        print("Số file Parquet nguồn:", len(urls))

        stream = load_dataset(
            "parquet",
            data_files={"train": urls},
            split="train",
            streaming=True
        )

        with RAW_FILE.open("a", encoding="utf-8") as output:
            for row in tqdm(
                stream,
                total=MAX_SCAN,
                desc="Đang quét"
            ):
                inspected += 1
                doc_id = str(row.get("doc_name") or "")

                if doc_id and doc_id not in seen_ids:
                    screening = screen_record(row)

                    if screening is not None:
                        digest = text_hash(row.get("markdown"))

                        if digest not in seen_hashes:
                            record = {
                                "source_dataset": DATASET_ID,
                                "source_revision": REVISION,
                                "text_sha256": digest,
                                "screening": screening,
                                "source_record": row
                            }

                            output.write(
                                json.dumps(
                                    record,
                                    ensure_ascii=False
                                ) + "\n"
                            )
                            output.flush()

                            seen_ids.add(doc_id)
                            seen_hashes.add(digest)

                            saved += 1
                            added += 1

                            if added % 25 == 0:
                                print(
                                    f"\nĐã quét {inspected:,} | "
                                    f"Tổng ứng viên {saved:,}"
                                )

                if saved >= TARGET:
                    print("\nĐã đạt số ứng viên mục tiêu.")
                    break

                if inspected >= MAX_SCAN:
                    print("\nĐã đạt giới hạn quét MAX_SCAN.")
                    break

    except KeyboardInterrupt:
        print(
            "\nĐã dừng. Các bản đã ghi vào file vẫn được giữ."
        )

    except Exception as error:
        print(
            "\nLượt quét bị lỗi:",
            type(error).__name__,
            str(error)[:1000]
        )
        print("Các bản đã lưu vẫn được giữ lại.")

    print(f"\nLượt này đã quét: {inspected:,}")
    print(f"Lượt này thêm: {added:,}")

print("Tổng số ứng viên đã lưu:", saved)
print("File:", RAW_FILE.resolve())

Đã lưu từ trước: 0
Số file Parquet nguồn: 16


Đang quét:   0%|          | 865/1542247 [01:13<3:12:44, 133.29it/s] 


Đã quét 753 | Tổng ứng viên 25


Đang quét:   0%|          | 1619/1542247 [01:14<48:59, 524.18it/s]  


Đã quét 1,553 | Tổng ứng viên 50


Đang quét:   0%|          | 2772/1542247 [01:16<32:01, 801.10it/s]

... (391 outputs left out)

Đang quét:   9%|▊         | 133614/1542247 [04:37<52:06, 450.54it/s]


Đã quét 133,524 | Tổng ứng viên 4,975


Đang quét:   9%|▊         | 134330/1542247 [04:39<48:51, 480.21it/s]  


Đã quét 134,331 | Tổng ứng viên 5,000

Đã đạt số ứng viên mục tiêu.

Lượt này đã quét: 134,331
Lượt này thêm: 5,000
Tổng số ứng viên đã lưu: 5000
File: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\notebooks\dataset_dan_su_dat_dai\candidates_raw.jsonl


In [8]:
if not RAW_FILE.exists():
    raise RuntimeError("Chưa có dữ liệu. Chạy ô 4 trước.")

records = []

with RAW_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

if not records:
    raise RuntimeError("Chưa tìm được ứng viên. Kiểm tra kết quả ô 4.")

MD_DIR = OUT / "ban_an_markdown"
MD_DIR.mkdir(exist_ok=True)

index_rows = []

for record in records:
    source = record["source_record"]
    screening = record["screening"]

    doc_id = str(source.get("doc_name") or record["text_sha256"][:16])
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", doc_id)[:150]
    text = str(source.get("markdown") or "")

    md_path = MD_DIR / f"{safe_id}.md"

    header = (
        f"# {source.get('official_document_id') or doc_id}\n\n"
        f"- Tòa án: {source.get('court')}\n"
        f"- Ngày: {source.get('issued_date')}\n"
        f"- Cấp xét xử: {source.get('instance_level')}\n"
        f"- Trang nguồn: {source.get('web_url')}\n"
        f"- PDF nguồn: {source.get('pdf_url')}\n\n"
        "---\n\n"
    )

    md_path.write_text(header + text, encoding="utf-8")

    index_rows.append({
        "doc_name": doc_id,
        "so_ban_an": source.get("official_document_id"),
        "ngay": source.get("issued_date"),
        "toa_an": source.get("court"),
        "linh_vuc": source.get("category"),
        "cap_xet_xu": source.get("instance_level"),
        "ma_loai": source.get("code"),
        "nhom_ung_vien": "; ".join(screening["candidate_groups"]),
        "tu_khoa": "; ".join(screening["matched_terms"]),
        "so_ky_tu": len(text),
        "web_url": source.get("web_url"),
        "pdf_url": source.get("pdf_url"),
        "file_markdown": md_path.name,
        # Các cột để người rà soát điền:
        "phu_hop_de_tai": "",
        "case_group_id": "",
        "co_du_yeu_cau": "",
        "co_ket_qua_so_tham": "",
        "du_kien_truoc_so_tham": "",
        "ghi_chu": ""
    })

df = pd.DataFrame(index_rows)
df.to_csv(
    OUT / "danh_muc_ra_soat.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df.head(10))

print("\nPhân bố cấp xét xử:")
print(df["cap_xet_xu"].value_counts(dropna=False))

print("\nSố tài liệu:", len(df))
print("Thư mục:", OUT.resolve())

# Nén để tải về hoặc gửi lại.
import shutil

zip_path = shutil.make_archive(
    str(OUT.resolve()),
    "zip",
    root_dir=OUT.resolve()
)

print("File ZIP:", zip_path)

# Chỉ chạy tải xuống tự động nếu đang ở Colab.
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("VS Code: mở thư mục trên để lấy file ZIP.")

,doc_name,so_ban_an,ngay,toa_an,linh_vuc,cap_xet_xu,ma_loai,nhom_ung_vien,tu_khoa,so_ky_tu,web_url,pdf_url,file_markdown,phu_hop_de_tai,case_group_id,co_du_yeu_cau,co_ket_qua_so_tham,du_kien_truoc_so_tham,ghi_chu
0,304,117/2017/DS-PT,2017-07-10,Tòa án nhân dân TỈNH TÂY NINH,Civil,Appellate,DS-PT,chuyen_nhuong_qsdd; dat_dai_khac,chuyen nhuong quyen su dung dat; hop dong chuy...,32237,https://congbobanan.toaan.gov.vn/2ta304t1cvn/c...,https://congbobanan.toaan.gov.vn/3ta304t1cvn/,304.md,,,,,,
1,672,1/2017/DS,2017-01-18,"Tòa án nhân dân THÀNH PHỐ L, TỈNH LÀO CAI CỘNG...",Civil,NaN,DS,dat_dai_khac,tranh chap dat dai; tranh chap quyen su dung dat,34124,https://congbobanan.toaan.gov.vn/2ta672t1cvn/c...,https://congbobanan.toaan.gov.vn/3ta672t1cvn/,672.md,,,,,,
2,880,11/2017/DSPT,2017-05-29,Tòa án nhân dân TỈNH HƯNG YÊN,Civil,Appellate,DS-PT,loi_di_bds_lien_ke; dat_dai_khac,mo loi di; tranh chap quyen su dung dat,24471,https://congbobanan.toaan.gov.vn/2ta880t1cvn/c...,https://congbobanan.toaan.gov.vn/3ta880t1cvn/,880.md,,,,,,
3,1744,10/2017/QĐST,2017-07-10,Tòa án nhân dân,NaN,First-instance,QĐST,chuyen_nhuong_qsdd,chuyen nhuong quyen su dung dat; hop dong chuy...,3965,https://congbobanan.toaan.gov.vn/2ta1744t1cvn/...,https://congbobanan.toaan.gov.vn/3ta1744t1cvn/,1744.md,,,,,,
4,2256,9/2017/DS-PT,2017-07-28,Tòa án nhân dân TỈNH YÊN BÁI,Civil,Appellate,DS-PT,dat_dai_khac,thua ke quyen su dung dat; tranh chap quyen su...,31556,https://congbobanan.toaan.gov.vn/2ta2256t1cvn/...,https://congbobanan.toaan.gov.vn/3ta2256t1cvn/,2256.md,,,,,,
5,3104,107/2017/QĐST-DS,2017-07-18,Tòa án nhân dân CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT...,Civil,First-instance,QĐST-DS,dat_dai_khac,tranh chap quyen su dung dat,2027,https://congbobanan.toaan.gov.vn/2ta3104t1cvn/...,https://congbobanan.toaan.gov.vn/3ta3104t1cvn/,3104.md,,,,,,
6,3680,20/2017/QĐST-DS,2017-07-31,Tòa án nhân dân,Civil,First-instance,QĐST-DS,dat_dai_khac,tranh chap quyen su dung dat,2964,https://congbobanan.toaan.gov.vn/2ta3680t1cvn/...,https://congbobanan.toaan.gov.vn/3ta3680t1cvn/,3680.md,,,,,,
7,4560,25/2017/DSPT,2017-08-02,Tòa án nhân dân TỈNH HẢI DƢƠNG,Civil,Appellate,DS-PT,ranh_gioi_lan_chiem,lan chiem dat,14496,https://congbobanan.toaan.gov.vn/2ta4560t1cvn/...,https://congbobanan.toaan.gov.vn/3ta4560t1cvn/,4560.md,,,,,,
8,5216,123/2017/DS-PT,2017-07-20,Tòa án nhân dân,Civil,Appellate,DS-PT,chuyen_nhuong_qsdd; ranh_gioi_lan_chiem,chuyen nhuong quyen su dung dat; dat lan chiem...,21341,https://congbobanan.toaan.gov.vn/2ta5216t1cvn/...,https://congbobanan.toaan.gov.vn/3ta5216t1cvn/,5216.md,,,,,,
9,5248,46/2017/DSST,2017-02-28,Tòa án nhân dân TỈNH VĨNH LONG,Civil,First-instance,DS-ST,chuyen_nhuong_qsdd,chuyen nhuong quyen su dung dat; hop dong chuy...,23592,https://congbobanan.toaan.gov.vn/2ta5248t1cvn/...,https://congbobanan.toaan.gov.vn/3ta5248t1cvn/,5248.md,,,,,,



Phân bố cấp xét xử:
cap_xet_xu
First-instance    2840
Appellate         2048
NaN                 58
Cassation           51
Retrial              3
Name: count, dtype: int64

Số tài liệu: 5000
Thư mục: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\notebooks\dataset_dan_su_dat_dai
File ZIP: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\notebooks\dataset_dan_su_dat_dai.zip
VS Code: mở thư mục trên để lấy file ZIP.
